In [1]:
# Kết nối đến Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [13]:
# 1. Import thư viện

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.mps.is_available()}")

root = "/Users/quangmanh/Project/Olympic-AI-Practice-1"

import warnings
warnings.filterwarnings("ignore")

PyTorch version: 2.8.0
MPS available: True


In [4]:
# 2. Thiết lập Seed để tái lập kết quả

def set_seed(seed=42):
    """
    Thiết lập seed cho tất cả các thư viện để đảm bảo kết quả có thể tái lập

    Args:
        seed (int): Giá trị seed cần thiết lập
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # Cho multi-GPU

    # Đảm bảo các thuật toán deterministic
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False

    # Thiết lập seed cho DataLoader workers
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"Đã thiết lập seed = {seed}")

SEED = 42
set_seed(SEED)

Đã thiết lập seed = 42


In [5]:
# 3. Định nghĩa Transform cho ảnh

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize về kích thước chuẩn
    transforms.ToTensor(),          # Chuyển sang tensor [0, 1]
    transforms.Normalize(           # Chuẩn hóa theo ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("Đã định nghĩa transform")

Đã định nghĩa transform


In [6]:
# 4. Xây dựng Dataset Class

class StormDataset(Dataset):
    """
    Dataset cho ảnh bão

    Cấu trúc thư mục:
        root_dir/
            ├── images/
            │   ├── image1.jpg
            │   └── ...
            └── annotations.csv (chỉ có trong folder train)

    Labels:
        - 0: Không bão (is_negative=True)
        - 1: TD (Tropical Depression)
        - 2: TS (Tropical Storm)
        - 3: STS (Severe Tropical Storm)
        - 4: TY (Typhoon)
    """
    def __init__(self, root_dir, transform=None, is_test=False):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "images")
        self.transform = transform
        self.is_test = is_test

        # Nếu là test set, lấy danh sách file từ thư mục images
        if is_test:
            self.image_files = sorted([f for f in os.listdir(self.img_dir)
                                      if f.endswith(('.jpg', '.jpeg', '.png'))])
            self.annotations = pd.DataFrame({'file_name': self.image_files})
        else:
            # Train set có file annotations.csv
            self.annotations = pd.read_csv(os.path.join(root_dir, "annotations.csv"))

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        img_path = os.path.join(self.img_dir, row['file_name'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Test set không có label
        if self.is_test:
            return image, -1  # Label giả cho test set

        # Gán nhãn cho train set
        if row['is_negative']:   # TRUE = không bão
            label = 0
        else:                    # FALSE = có bão
            label = int(row['category_id'])  # TD=1, TS=2, STS=3, TY=4

        return image, label

print("Đã định nghĩa StormDataset")

Đã định nghĩa StormDataset


In [7]:
# 5. Xây dựng Model CNN

class SimpleCNN(nn.Module):
    """
    Mô hình CNN đơn giản cho phân loại bão

    Architecture:
        - Conv1: 3 -> 32 channels, MaxPool
        - Conv2: 32 -> 64 channels
        - Global Average Pooling
        - Fully Connected: 64 -> num_classes
    """
    def __init__(self, num_classes=5, in_ch=3):
        super().__init__()

        # Block 1: Conv + BatchNorm + ReLU + MaxPool
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # Giảm 1/2 kích thước H, W
        )

        # Block 2: Conv + BatchNorm + ReLU
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Global Average Pooling: (B, 64, H, W) -> (B, 64, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layer
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv1(x)           # (B, 32, H/2, W/2)
        x = self.conv2(x)           # (B, 64, H/2, W/2)
        x = self.gap(x).flatten(1)  # (B, 64)
        x = self.fc(x)              # (B, num_classes)
        return x

# Test model
model = SimpleCNN()
print(f"Đã xây dựng SimpleCNN")
print(f"Tổng số parameters: {sum(p.numel() for p in model.parameters()):,}")

Đã xây dựng SimpleCNN
Tổng số parameters: 19,813


In [17]:
# 6. Chuẩn bị DataLoader

def worker_init_fn(worker_id):
    """
    Hàm khởi tạo seed cho mỗi worker trong DataLoader
    Đảm bảo tính deterministic khi load data
    """
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Load dataset
full_dataset = StormDataset(os.path.join(root, "data/train"), transform=transform)
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

print(f"Tổng số ảnh: {total_size}")
print(f"Train: {train_size}, Validation: {val_size}")

# Chia train/val với seed cố định
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

# Tạo DataLoader
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    worker_init_fn=worker_init_fn,
    num_workers=0,
    pin_memory=True
)

print(f"Đã tạo DataLoader (batch_size={BATCH_SIZE})")

Tổng số ảnh: 8385
Train: 6708, Validation: 1677
Đã tạo DataLoader (batch_size=64)


In [18]:
# 7. Huấn luyện Model

# Thiết lập device
device = torch.device('mps') if torch.mps.is_available() else torch.device('cpu')
print(f"🔧 Sử dụng device: {device}")

# Khởi tạo model, loss, optimizer
model = SimpleCNN(num_classes=5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)

# Hyperparameters
NUM_EPOCHS = 5

print(f"Bắt đầu huấn luyện ({NUM_EPOCHS} epochs)...")

# Lưu history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

# %%
# Training loop
for epoch in range(NUM_EPOCHS):
    # ===== TRAINING PHASE =====
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Thống kê
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = 100 * correct / total

    # ===== VALIDATION PHASE =====
    model.eval()
    val_correct, val_total = 0, 0
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_acc = 100 * val_correct / val_total

    # Lưu history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # In kết quả
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%\n")

print("Hoàn thành huấn luyện!")

🔧 Sử dụng device: mps
Bắt đầu huấn luyện (5 epochs)...


Epoch 1/5 [Train]:   0%|          | 0/105 [00:00<?, ?it/s]

Epoch 1/5 [Val]: 100%|██████████| 27/27 [00:07<00:00,  3.49it/s]



Epoch [1/5]:
  Train Loss: 1.2017, Train Acc: 49.55%
  Val Loss: 1.1374, Val Acc: 51.10%



Epoch 2/5 [Val]: 100%|██████████| 27/27 [00:07<00:00,  3.54it/s]


...


Epoch [3/5]:
  Train Loss: 1.0319, Train Acc: 54.64%
  Val Loss: 1.3412, Val Acc: 40.97%



Epoch 4/5 [Val]: 100%|██████████| 27/27 [00:07<00:00,  3.50it/s]



Epoch [4/5]:
  Train Loss: 0.9904, Train Acc: 55.26%
  Val Loss: 1.2716, Val Acc: 45.08%



Epoch 5/5 [Val]: 100%|██████████| 27/27 [00:07<00:00,  3.50it/s]


Epoch [5/5]:
  Train Loss: 0.9928, Train Acc: 56.13%
  Val Loss: 1.5952, Val Acc: 40.13%

Hoàn thành huấn luyện!


In [ ]:
# 8. Lưu Model

# Lưu model weights
model_path = os.path.join(root, "cnn_model_pytorch.pth")
torch.save(model.state_dict(), model_path)
print(f"Đã lưu model tại: {model_path}")

# Lưu history
history_path = os.path.join(root, "training_history.csv")
history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print(f"Đã lưu lịch sử huấn luyện tại: {history_path}")

In [ ]:
# 9. Inference trên Public Test Set

print("Bắt đầu inference trên PUBLIC TEST SET...")

# Load test dataset
public_test_dataset = StormDataset(os.path.join(root, "dataset/public_test"), transform=transform, is_test=True)
public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inference
model.eval()
file_names = []
preds_is_negative = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(public_test_loader, desc="Public Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        # Lấy file names
        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = public_test_dataset.annotations.iloc[batch_start:batch_end]['file_name'].values
        file_names.extend(batch_files)

        # Chuyển đổi predictions
        # label = 0 → không bão (is_negative=True)
        # label = 1-4 → có bão (is_negative=False, category_id = label)
        for p in predicted.cpu().numpy():
            if p == 0:
                preds_is_negative.append(True)
                preds_category_id.append("")
            else:
                preds_is_negative.append(False)
                preds_category_id.append(int(p))

# Tạo DataFrame kết quả
df_public = pd.DataFrame({
    'file_name': file_names,
    'is_negative': preds_is_negative,
    'category_id': preds_category_id
})

# Lưu file
public_csv_path = os.path.join(root, "public_cv.csv")
df_public.to_csv(public_csv_path, index=False)
print(f"Đã tạo file public_cv.csv tại {public_csv_path} với {len(df_public)} dự đoán!")

# Hiển thị preview
print("\nPreview kết quả:")
print(df_public.head(10))

In [ ]:
# 10. Inference trên Private Test Set

print("Bắt đầu inference trên PRIVATE TEST SET...")

# Load test dataset
private_test_dataset = StormDataset(os.path.join(root, "dataset/private_test"), transform=transform, is_test=True)
private_test_loader = DataLoader(
    private_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Inference
model.eval()
file_names = []
preds_is_negative = []
preds_category_id = []

with torch.no_grad():
    for i, (images, _) in enumerate(tqdm(private_test_loader, desc="Private Test")):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        # Lấy file names
        batch_start = i * BATCH_SIZE
        batch_end = batch_start + images.size(0)
        batch_files = private_test_dataset.annotations.iloc[batch_start:batch_end]['file_name'].values
        file_names.extend(batch_files)

        # Chuyển đổi predictions
        for p in predicted.cpu().numpy():
            if p == 0:
                preds_is_negative.append(True)
                preds_category_id.append("")
            else:
                preds_is_negative.append(False)
                preds_category_id.append(int(p))

# Tạo DataFrame kết quả
df_private = pd.DataFrame({
    'file_name': file_names,
    'is_negative': preds_is_negative,
    'category_id': preds_category_id
})

# Lưu file
private_csv_path = os.path.join(root, "private_cv.csv")
df_private.to_csv(private_csv_path, index=False)
print(f"Đã tạo file private_cv.csv với {len(df_private)} dự đoán!")

# Hiển thị preview
print("\nPreview kết quả:")
print(df_private.head(10))